# ESM2 + DoRA + PSSM（seed=44）SmallHead 重训与外部验证

**配置**：ESM2 backbone + DoRA adapter（lm_pssm变体，seed=44）  
**流程**：
1. 加载冻结 backbone + adapter，重训 SmallHead
2. 在原始 test set 验证 AUC / AUPRC
3. 对 11 条 new_case 外部蛋白提取特征并输出预测概率

---

In [ ]:
import json
import os
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from peft import PeftModel
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

def _find_repo_root(start: Path) -> Path:
    cur = start.resolve()
    for p in [cur, *cur.parents]:
        if (p / "scripts" / "main.py").exists() and (p / "src").exists():
            return p
    raise FileNotFoundError("Cannot locate repository root containing scripts/main.py and src/")

REPO_ROOT = (
    Path(os.environ["ACRPLMEVO_REPO_ROOT"]).expanduser().resolve()
    if os.environ.get("ACRPLMEVO_REPO_ROOT")
    else _find_repo_root(Path.cwd())
)
SCRIPTS_ROOT = REPO_ROOT / "scripts"
SRC_ROOT = REPO_ROOT / "src"
for _p in [str(SCRIPTS_ROOT), str(SRC_ROOT)]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

from main import (  # type: ignore
    BACKBONE_SPECS,
    SequenceBinaryClassifier,
    SequenceDataset,
    TrainConfig,
    get_autocast_dtype,
    get_device,
    load_split_data,
    load_tokenizer_and_model,
    make_collator,
)
from acrplmevo.pssm_fusion import (  # type: ignore
    attach_pssm_features,
    evaluate_binary,
    find_best_threshold,
    load_feature_cache,
)

print("Imports OK")

/home/nemophila/miniconda3/envs/lm-hf/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-03-31 14:47:45.840784: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-31 14:47:45.890324: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-31 14:47:47.333157: I tensorflow/c

Imports OK


In [ ]:
# ─── 配置常量 ────────────────────────────────────────────────────────────────────────────
SEED           = 44
MODEL_NAME     = "esm2"
ADAPTER_TYPE   = "dora"
VARIANT        = "lm_pssm"   # adapter 变体 & 特征变体（全程一致）

HEAD_EPOCHS    = 40
HEAD_LR        = 1e-3
HEAD_BATCH     = 64
PATIENCE       = 3
DROPOUT        = 0.3
PCA_DIM        = 128

# 阈值策略：在验证集上选择满足 Recall ≥ TARGET_RECALL 的阈值（应用场景：外部注释集全为 Acr 蛋白，漏检代价高于误报）
TARGET_RECALL  = 0.95

ADAPTER_RUNS_ROOT = Path(os.environ.get("ACRPLMEVO_RUNS_ROOT", str(REPO_ROOT / "results" / "runs")))
BENCHMARKS_DIR = Path(os.environ.get("ACRPLMEVO_EXTERNAL_BENCHMARKS_DIR", str(REPO_ROOT / "external_validation" / "data")))
PSSM_WORK_ROOT = Path(os.environ.get("PSSM_WORK_ROOT", str(REPO_ROOT / "data" / "pssm_work")))
NEW_CASE_PSSM_ROOT = Path(os.environ.get("NEW_CASE_PSSM_ROOT", str(REPO_ROOT / "data" / "new_case_pssm_work")))

ADAPTER_DIR  = ADAPTER_RUNS_ROOT / ADAPTER_TYPE / MODEL_NAME / VARIANT / f"seed_{SEED}" / "adapter"
NEW_CASE_CSV  = BENCHMARKS_DIR / "new_case.csv"

assert ADAPTER_DIR.exists(), f"Adapter not found: {ADAPTER_DIR}"
assert NEW_CASE_CSV.exists(), f"new_case.csv not found: {NEW_CASE_CSV}"
print(f"Adapter:   {ADAPTER_DIR}")
print(f"New case:  {NEW_CASE_CSV}")
print(f"Threshold strategy: High-recall (target={TARGET_RECALL}) selected on validation set")

Adapter:   results/runs/dora/esm2/lm_pssm/seed_44/adapter
New case:  external_validation/data/new_case.csv
Threshold strategy: High-recall (target=0.95) selected on validation set


In [3]:
# ─── 工具函数（与 run_frozen_baseline_all.py 保持一致） ──────────────────────

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def sanitize(x: np.ndarray) -> np.ndarray:
    return np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)


class SmallHead(nn.Module):
    def __init__(self, in_dim: int, dropout: float) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(in_dim),
            nn.Linear(in_dim, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


def extract_features(df, tokenizer, backbone, spec, device):
    collate_fn = make_collator(tokenizer, spec.seq_mode, spec.max_length)
    dataset = SequenceDataset(df, [])
    loader  = DataLoader(dataset, batch_size=spec.batch_size, shuffle=False, collate_fn=collate_fn)
    autocast_dtype = get_autocast_dtype(device)

    feats, labels, ids = [], [], []
    backbone.eval()
    for raw_batch in loader:
        kwargs = {
            "input_ids":      raw_batch["input_ids"].to(device),
            "attention_mask": raw_batch["attention_mask"].to(device),
            "return_dict":    True,
        }
        if "token_type_ids" in raw_batch:
            kwargs["token_type_ids"] = raw_batch["token_type_ids"].to(device)
        with torch.no_grad():
            with torch.autocast(device_type=device.type, dtype=autocast_dtype,
                                enabled=autocast_dtype is not None):
                out = backbone(**kwargs)
            pooled = SequenceBinaryClassifier.masked_mean(
                out.last_hidden_state, kwargs["attention_mask"]
            ).detach().cpu().float().numpy()
        ids.extend(list(raw_batch["sample_ids"]))
        labels.append(raw_batch["labels"].numpy().astype(np.int32))
        feats.append(pooled)
    return sanitize(np.concatenate(feats)), np.concatenate(labels), ids


def fit_pca(x_tr, x_va, x_te, seed):
    x_tr, x_va, x_te = sanitize(x_tr), sanitize(x_va), sanitize(x_te)
    n = min(PCA_DIM, x_tr.shape[1], x_tr.shape[0])
    pca = PCA(n_components=n, random_state=seed)
    tr = pca.fit_transform(x_tr)
    va = pca.transform(x_va)
    te = pca.transform(x_te)
    pad = PCA_DIM - n
    if pad > 0:
        tr = np.pad(tr, ((0,0),(0,pad)))
        va = np.pad(va, ((0,0),(0,pad)))
        te = np.pad(te, ((0,0),(0,pad)))
    return tr.astype(np.float32), va.astype(np.float32), te.astype(np.float32), pca


def train_head(x_tr, y_tr, x_va, y_va, device):
    head = SmallHead(x_tr.shape[1], DROPOUT).to(device)
    opt  = torch.optim.Adam(head.parameters(), lr=HEAD_LR)
    crit = nn.BCEWithLogitsLoss()
    tr_loader = DataLoader(TensorDataset(torch.from_numpy(x_tr),
                                         torch.from_numpy(y_tr.astype(np.float32))),
                           batch_size=HEAD_BATCH, shuffle=True)
    va_loader = DataLoader(TensorDataset(torch.from_numpy(x_va),
                                         torch.from_numpy(y_va.astype(np.float32))),
                           batch_size=HEAD_BATCH, shuffle=False)
    best_val, best_state, bad = float("inf"), None, 0
    for epoch in range(1, HEAD_EPOCHS + 1):
        head.train()
        for xb, yb in tr_loader:
            opt.zero_grad(set_to_none=True)
            crit(head(xb.to(device)), yb.to(device)).backward()
            opt.step()
        head.eval()
        val_loss = float(np.mean([crit(head(xb.to(device)), yb.to(device)).item()
                                  for xb, yb in va_loader]))
        print(f"  epoch {epoch:2d}  val_loss={val_loss:.4f}")
        if val_loss < best_val:
            best_val = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in head.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= PATIENCE:
                print(f"  Early stop at epoch {epoch}")
                break
    head.load_state_dict(best_state)
    return head


def predict_prob(head, x, device):
    head.eval()
    loader = DataLoader(TensorDataset(torch.from_numpy(x.astype(np.float32))),
                        batch_size=HEAD_BATCH, shuffle=False)
    probs = []
    with torch.no_grad():
        for (xb,) in loader:
            probs.append(torch.sigmoid(head(xb.to(device))).cpu().numpy())
    return np.concatenate(probs).astype(np.float32)


def _binary_stats(y_true, y_prob, thr):
    y_pred = (y_prob >= thr).astype(int)
    tp = int(np.sum((y_pred == 1) & (y_true == 1)))
    fp = int(np.sum((y_pred == 1) & (y_true == 0)))
    fn = int(np.sum((y_pred == 0) & (y_true == 1)))
    tn = int(np.sum((y_pred == 0) & (y_true == 0)))
    recall = tp / (tp + fn + 1e-12)
    precision = tp / (tp + fp + 1e-12)
    return {"tp": tp, "fp": fp, "fn": fn, "tn": tn, "recall": recall, "precision": precision}


def find_threshold_high_recall(y_true, y_prob, target_recall=0.90, grid=None):
    if grid is None:
        grid = np.linspace(0.05, 0.95, 19)
    grid = sorted(float(t) for t in grid)

    # 在满足 recall >= target 的候选中，选“最高阈值”（更保守，降低FP）
    candidates = []
    for thr in grid:
        s = _binary_stats(y_true, y_prob, thr)
        if s["recall"] >= target_recall:
            candidates.append((thr, s["recall"], s["precision"]))

    if len(candidates) > 0:
        best = max(candidates, key=lambda x: (x[0], x[2]))  # 先高阈值，再高precision
        return float(best[0]), {"mode": "target_recall_met", "recall": best[1], "precision": best[2]}

    # 若无候选满足target，退化为“最大recall，再最大precision，再最低阈值”
    scored = []
    for thr in grid:
        s = _binary_stats(y_true, y_prob, thr)
        scored.append((thr, s["recall"], s["precision"]))
    best = max(scored, key=lambda x: (x[1], x[2], -x[0]))
    return float(best[0]), {"mode": "fallback_max_recall", "recall": best[1], "precision": best[2]}


print("Helper functions defined.")

Helper functions defined.


## Phase 1：加载训练数据与 PSSM 特征

In [ ]:
set_seed(SEED)

# 关键自检：确保主缓存仍是 train_/test_ 的完整缓存，而不是被 new_case(exval_) 覆盖
main_cache = PSSM_WORK_ROOT / "features" / "pssm_features_1110.csv"
assert main_cache.exists(), f"Missing main cache: {main_cache}"
cache_sid = pd.read_csv(main_cache, usecols=["sample_id"])
n_train = cache_sid["sample_id"].astype(str).str.startswith("train_").sum()
n_test  = cache_sid["sample_id"].astype(str).str.startswith("test_").sum()
n_exval = cache_sid["sample_id"].astype(str).str.startswith("exval_").sum()
print(f"Main cache rows={len(cache_sid)} | train={n_train} test={n_test} exval={n_exval}")
assert n_train > 900 and n_test > 200, (
    "Main pssm cache looks wrong (likely overwritten by new_case). "
    f"Please rebuild {main_cache} using sample_manifest.csv before running this notebook."
)

spec = BACKBONE_SPECS[MODEL_NAME]

# TrainConfig 仅用于 load_split_data 的参数透传
train_cfg = TrainConfig(
    seed=SEED, model_name=MODEL_NAME, variant=VARIANT,
    batch_size=spec.batch_size, max_length=spec.max_length,
    adapter_type=ADAPTER_TYPE,
    max_train_samples=None, max_valid_samples=None, max_test_samples=None,
)

train_df, valid_df, test_df, feature_cols = load_split_data(VARIANT, SEED, train_cfg)

print(f"train={len(train_df)}  valid={len(valid_df)}  test={len(test_df)}")
print(f"PSSM feature columns: {len(feature_cols)}")

Main cache rows=1393 | train=1107 test=286 exval=0
train=996  valid=111  test=286
PSSM feature columns: 1110


## Phase 2：加载冻结 ESM2 backbone + DoRA adapter

In [5]:
device = get_device()
print(f"Device: {device}")

model_id, tokenizer, backbone, _, _ = load_tokenizer_and_model(spec)
backbone = PeftModel.from_pretrained(backbone, str(ADAPTER_DIR), is_trainable=False)
backbone = backbone.to(device)
backbone.eval()

trainable = sum(p.numel() for p in backbone.parameters() if p.requires_grad)
total     = sum(p.numel() for p in backbone.parameters())
print(f"Model:     {model_id}")
print(f"Trainable: {trainable:,}  /  Total: {total:,}")

Device: cuda


Loading weights: 100%|██████████| 617/617 [00:00<00:00, 1396.38it/s]
EsmModel LOAD REPORT from: facebook/esm2_t36_3B_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.decoder.weight      | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
pooler.dense.bias           | MISSING    | 
pooler.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model:     facebook/esm2_t36_3B_UR50D
Trainable: 0  /  Total: 2,842,133,921


## Phase 3：提取 LM 特征 + 拼接 PSSM + PCA → 训练 SmallHead

In [6]:
print("Extracting LM features (train)...")
x_tr_lm, y_tr, train_ids = extract_features(train_df, tokenizer, backbone, spec, device)
print("Extracting LM features (valid)...")
x_va_lm, y_va, valid_ids = extract_features(valid_df, tokenizer, backbone, spec, device)
print("Extracting LM features (test)...")
x_te_lm, y_te, test_ids  = extract_features(test_df,  tokenizer, backbone, spec, device)

print(f"LM feature dim: {x_tr_lm.shape[1]}")

Extracting LM features (train)...
Extracting LM features (valid)...
Extracting LM features (test)...
LM feature dim: 2560


In [7]:
# 拼接 PSSM 特征（已在 load_split_data 中经过 StandardScaler）
pssm_tr = sanitize(train_df[feature_cols].values.astype(np.float32))
pssm_va = sanitize(valid_df[feature_cols].values.astype(np.float32))
pssm_te = sanitize(test_df[feature_cols].values.astype(np.float32))

x_tr_raw = np.concatenate([x_tr_lm, pssm_tr], axis=1)
x_va_raw = np.concatenate([x_va_lm, pssm_va], axis=1)
x_te_raw = np.concatenate([x_te_lm, pssm_te], axis=1)

print(f"Combined dim before PCA: {x_tr_raw.shape[1]}")

# 拟合 PCA（保存 pca 对象，后续用于 new_case 投影）
x_tr, x_va, x_te, pca_model = fit_pca(x_tr_raw, x_va_raw, x_te_raw, SEED)
print(f"PCA output dim: {x_tr.shape[1]}")

Combined dim before PCA: 3670
PCA output dim: 128


In [8]:
print("Training SmallHead...")
head = train_head(x_tr, y_tr, x_va, y_va, device)

Training SmallHead...
  epoch  1  val_loss=0.4876
  epoch  2  val_loss=0.3785
  epoch  3  val_loss=0.3101
  epoch  4  val_loss=0.2607
  epoch  5  val_loss=0.2227
  epoch  6  val_loss=0.1960
  epoch  7  val_loss=0.1796
  epoch  8  val_loss=0.1713
  epoch  9  val_loss=0.1628
  epoch 10  val_loss=0.1591
  epoch 11  val_loss=0.1558
  epoch 12  val_loss=0.1538
  epoch 13  val_loss=0.1548
  epoch 14  val_loss=0.1575
  epoch 15  val_loss=0.1593
  Early stop at epoch 15


## Phase 4：在原始 test set 上进行双协议评估（A: F1-opt，B: High-recall）

In [9]:
# 在 valid 集上选择高召回率阈值，在 test 集评估指标
va_prob = predict_prob(head, x_va, device)
te_prob = predict_prob(head, x_te, device)

# 阈值选择：在验证集上找满足 Recall ≥ TARGET_RECALL 的最高阈值（降低误报）
best_thr, thr_meta = find_threshold_high_recall(y_va, va_prob, target_recall=TARGET_RECALL)

va_stats = _binary_stats(y_va, va_prob, best_thr)
te_metrics = evaluate_binary(y_te, te_prob, threshold=best_thr)
te_stats = _binary_stats(y_te, te_prob, best_thr)

print(f"\n=== Threshold Selection (valid set, target recall={TARGET_RECALL}) ===")
print(f"  Threshold:       {best_thr:.2f}  (mode: {thr_meta['mode']})")
print(f"  Valid recall:    {va_stats['recall']:.4f}")
print(f"  Valid precision: {va_stats['precision']:.4f}")
print(f"\n=== Internal Test Set Performance ===")
print(f"  AUC={te_metrics['AUC']:.4f}  AUPRC={te_metrics['AUPRC']:.4f}")
print(f"  F1={te_metrics['F1']:.4f}  MCC={te_metrics['MCC']:.4f}")
print(f"  Recall={te_stats['recall']:.4f}  Precision={te_stats['precision']:.4f}")
print(f"\nNote: AUC/AUPRC are threshold-independent; threshold affects recall/precision/F1/MCC only.")



=== Threshold Selection (valid set, target recall=0.95) ===
  Threshold:       0.15  (mode: target_recall_met)
  Valid recall:    0.9524
  Valid precision: 0.6897

=== Internal Test Set Performance ===
  AUC=0.9648  AUPRC=0.7776
  F1=0.6410  MCC=0.6393
  Recall=0.9615  Precision=0.4808

Note: AUC/AUPRC are threshold-independent; threshold affects recall/precision/F1/MCC only.


## Phase 5：对 new_case 外部蛋白进行预测

In [10]:
# 加载 new_case 序列
new_case_df = pd.read_csv(NEW_CASE_CSV)
new_case_df["sample_id"] = [f"exval_{i:06d}" for i in range(len(new_case_df))]
new_case_df["label"] = new_case_df["label"].astype(int)

# 加载 new_case PSSM 特征（使用发布时相同的 feature 列名）
nc_pssm_cache = NEW_CASE_PSSM_ROOT / "features" / "pssm_features_1110.csv"
nc_pssm_df, nc_feat_cols = load_feature_cache(str(nc_pssm_cache))

# 对齐 feature_cols（与训练时的 1110 维列名保持一致）
# new_case cache 的 feat_* 列名应与训练集相同，按位置对应
assert len(nc_feat_cols) == len(feature_cols), (
    f"PSSM dim mismatch: new_case={len(nc_feat_cols)}, train={len(feature_cols)}"
)

# 重命名 nc_pssm_df 的 feat_* 列，使其与训练集 feature_cols 对齐
nc_pssm_df = nc_pssm_df.rename(columns=dict(zip(nc_feat_cols, feature_cols)))

# 将 PSSM 附加到 new_case_df
new_case_df = attach_pssm_features(new_case_df, nc_pssm_df, feature_cols)

print(f"New case sequences: {len(new_case_df)}")
print(new_case_df[["sample_id", "label"]].to_string(index=False))

New case sequences: 11
   sample_id  label
exval_000000      1
exval_000001      1
exval_000002      1
exval_000003      1
exval_000004      1
exval_000005      1
exval_000006      1
exval_000007      1
exval_000008      1
exval_000009      1
exval_000010      1


In [11]:
# 对 new_case PSSM 应用与训练集相同的 StandardScaler
# 注意：scaler 已在 load_split_data 内部 fit_transform；此处需手动重建
# 从原始（未缩放）训练集 PSSM 重新 fit scaler
from llm_lora_benchmark import pick_pssm_cache  # type: ignore
from proteinbert.pssm_fusion import load_anticrispr_with_ids  # type: ignore
from sklearn.model_selection import train_test_split

raw_train_pool, _ = load_anticrispr_with_ids(str(BENCHMARKS_DIR))
raw_feat_df, raw_feat_cols = load_feature_cache(str(pick_pssm_cache()))
raw_train_pool = attach_pssm_features(raw_train_pool, raw_feat_df, raw_feat_cols)
raw_train_df, _ = train_test_split(
    raw_train_pool, test_size=0.1,
    stratify=raw_train_pool["label"], random_state=SEED
)

scaler = StandardScaler()
scaler.fit(raw_train_df[raw_feat_cols].to_numpy(dtype=np.float32))

# 对 new_case PSSM 进行缩放
nc_pssm_scaled = scaler.transform(
    sanitize(new_case_df[feature_cols].values.astype(np.float32))
)
print(f"New case PSSM scaled: {nc_pssm_scaled.shape}")

New case PSSM scaled: (11, 1110)


In [12]:
print("Extracting LM features for new_case...")
x_nc_lm, _, nc_ids = extract_features(new_case_df, tokenizer, backbone, spec, device)
print(f"LM features shape: {x_nc_lm.shape}")

Extracting LM features for new_case...
LM features shape: (11, 2560)


In [13]:
# 拼接 PSSM → PCA 投影（使用训练时 fit 的 pca_model）→ SmallHead 预测
x_nc_raw = np.concatenate([x_nc_lm, nc_pssm_scaled], axis=1)
x_nc     = sanitize(pca_model.transform(sanitize(x_nc_raw))).astype(np.float32)
if x_nc.shape[1] < PCA_DIM:
    x_nc = np.pad(x_nc, ((0,0),(0, PCA_DIM - x_nc.shape[1])))

nc_prob = predict_prob(head, x_nc, device)
nc_pred = (nc_prob >= best_thr).astype(int)

print(f"Threshold: {best_thr:.2f}  (High-recall strategy, target={TARGET_RECALL}, selected on valid set)")
print(f"Shape: {x_nc.shape}")


Threshold: 0.15  (High-recall strategy, target=0.95, selected on valid set)
Shape: (11, 128)


## Phase 6：预测结果汇总

In [14]:
result_df = pd.DataFrame({
    "sample_id":  new_case_df["sample_id"].values,
    "seq":        new_case_df["seq"].values,
    "y_true":     new_case_df["label"].values,
    "y_prob":     np.round(nc_prob.astype(float), 4),
    "y_pred":     nc_pred,
    "threshold":  best_thr,
})

# 按预测概率降序展示
result_df = result_df.sort_values("y_prob", ascending=False).reset_index(drop=True)

print(f"{'='*72}")
print(f"  New Case External Validation Results")
print(f"  Model: ESM2 + DoRA + PSSM (lm_pssm, seed=44)")
print(f"  Threshold: {best_thr:.2f}  (High-recall, target={TARGET_RECALL}, selected on valid set)")
print(f"{'='*72}")
print(result_df[["sample_id","y_prob","y_pred","y_true"]].to_string(index=False))

n_correct = (result_df["y_pred"] == result_df["y_true"]).sum()
n_total   = len(result_df)
print(f"\n  Predicted positive (Anti-CRISPR): {result_df['y_pred'].sum()} / {n_total}")
print(f"  Correct predictions:              {n_correct} / {n_total}")


  New Case External Validation Results
  Model: ESM2 + DoRA + PSSM (lm_pssm, seed=44)
  Threshold: 0.15  (High-recall, target=0.95, selected on valid set)
   sample_id  y_prob  y_pred  y_true
exval_000004  0.9661       1       1
exval_000002  0.8803       1       1
exval_000003  0.8768       1       1
exval_000001  0.8212       1       1
exval_000009  0.5232       1       1
exval_000006  0.4045       1       1
exval_000005  0.2194       1       1
exval_000008  0.1753       1       1
exval_000007  0.1733       1       1
exval_000000  0.1529       1       1
exval_000010  0.1094       0       1

  Predicted positive (Anti-CRISPR): 10 / 11
  Correct predictions:              10 / 11


In [ ]:
# 保存结果到 CSV
OUT_CSV = REPO_ROOT / "external_validation" / "results" / "new_case_predictions_esm2_dora_pssm_seed44.csv"
result_df.to_csv(OUT_CSV, index=False)
print(f"Saved to: {OUT_CSV}")

Saved to: external_validation/results/new_case_predictions_esm2_dora_pssm_seed44.csv
